In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# OpenAI Responses API를 활용한 서비스 구현 튜토리얼

본 튜토리얼에서는 OpenAI **Responses API**를 활용하여 AI 기반 서비스를 개발하는 방법을 단계별로 설명합니다.

## 1. OpenAI Responses API 소개

Responses API는 OpenAI가 제공하는 최신 API로, 모델에게 역할과 질문을 함께 전달하면 바로 답변을 받을 수 있습니다. 다음과 같은 특징이 있습니다.

- **대화 지속성**: 직전 응답의 `response.id`를 `previous_response_id`로 넘기기만 하면, 별도의 저장 공간을 만들지 않고도 이전 대화 맥락이 자동으로 이어집니다.
- **다양한 도구 통합**: 코드 실행(Code Interpreter), 문서 검색(File Search), 웹 검색(Web Search), 함수 호출(Function Calling) 등의 도구를 모델에 연결할 수 있습니다. 이를 통해 모델의 한계를 넘어서는 작업(예: 데이터 계산, 파일 처리, 외부 검색 등)을 자동화할 수 있습니다.
- **개인화된 지시어**: `instructions` 파라미터로 모델의 성격과 역할을 정의할 수 있습니다. 예를 들어 "당신은 친절한 고객지원 봇입니다"와 같은 지시어로 모델의 톤과 도메인 지식을 설정할 수 있습니다.
- **간결한 응답 접근**: `response.output_text`로 최종 텍스트 답변을 바로 꺼낼 수 있습니다.

요약하면, Responses API는 복잡한 대화 상태 관리, 외부 도구 통합, 문서 검색 등 많은 부분을 OpenAI 플랫폼이 맡아주므로, 개발자는 핵심 로직 구현에 집중할 수 있습니다. 이러한 이유로, 챗봇이나 자동화 에이전트를 만든다면 Responses API가 강력한 선택지가 됩니다.

>Note: API는 계속 발전하므로, 사용 전 [공식 문서](https://platform.openai.com/docs/api-reference/responses)를 확인하는 습관을 들이는 게 좋습니다.

## 2. API 키 설정 및 환경 변수 사용

OpenAI API를 사용하려면 API 키가 필요합니다. OpenAI 플랫폼의 API 키 관리 페이지에서 비밀 키를 생성할 수 있습니다. 생성된 키는 한 번만 표시되므로, 반드시 복사하여 안전한 곳에 저장하세요. 일반적으로 이 키를 소스 코드에 하드코딩하지 않고, 별도의 설정으로 관리하는 것이 좋습니다. **환경 변수(Environment Variable)**를 사용하면 API 키를 소스 코드에 노출하지 않고 관리할 수 있습니다. 개발 PC 또는 서버의 환경 변수 OPENAI_API_KEY에 키를 저장해 두면, OpenAI 라이브러리가 자동으로 이를 읽어 사용할 수 있습니다. Python 개발 환경에서는 python-dotenv 패키지를 활용해 .env 파일에 키를 저장하고 로드하는 방식이 편리합니다.

다음은 API 키를 설정하고 로드하는 과정입니다:

1. python-dotenv 설치: 터미널에서 `pip install python-dotenv` 명령으로 설치합니다 (한번만 수행).

2. 환경 변수 파일 생성: 프로젝트 루트 디렉토리에 .env 파일을 만들고, 아래와 같이 OpenAI API 키를 입력합니다 (따옴표 없이 실제 키로 대체).

    ```
    OPENAI_API_KEY=sk-***********************
    ```

3. 코드에서 로드: Python 코드에서 python-dotenv를 이용해 .env를 로드합니다. OpenAI 공식 Python SDK는 환경 변수 OPENAI_API_KEY를 자동으로 인식하므로, `OpenAI()`를 인자 없이 호출해도 내부적으로 이 값을 사용합니다.
M

In [2]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
client = OpenAI()